In [10]:
from pathlib import Path
from collections import Counter, defaultdict
import random
import shutil

SEED = 42
VAL_FRAC = 0.10
MODALITY = 'hsi'  # change to 'rgb' if needed

In [11]:
!pwd

/home/echerif/ironhack/neon_tree_crown_v3/notebooks


In [12]:
BASE = Path('/mnt/c/users/eya/documents/echerif_PhD/IronHack/NEON_data_24Sites')

if MODALITY == 'rgb':
    img_dir = BASE / 'rgb_patches_tr_nooverlap' / 'img'
    lbl_dir = BASE / 'rgb_patches_tr_nooverlap' / 'labels'
    out_root = Path('../data/splits/rgb_binary_10pct_balanced')
elif MODALITY == 'hsi':
    img_dir = BASE / 'hsi_patches_tr_nooverlap' / 'img'
    lbl_dir = BASE / 'hsi_patches_tr_nooverlap' / 'labels'
    out_root = Path('../data/splits/hsi_binary_10pct_balanced')
else:
    raise ValueError(f'Unsupported modality: {MODALITY}')

random.seed(SEED)
print('img_dir =', img_dir)
print('lbl_dir =', lbl_dir)
print('out_root =', out_root)
print('VAL_FRAC =', VAL_FRAC)

img_dir = /mnt/c/users/eya/documents/echerif_PhD/IronHack/NEON_data_24Sites/hsi_patches_tr_nooverlap/img
lbl_dir = /mnt/c/users/eya/documents/echerif_PhD/IronHack/NEON_data_24Sites/hsi_patches_tr_nooverlap/labels
out_root = ../data/splits/hsi_binary_10pct_balanced
VAL_FRAC = 0.1


In [13]:
def year_site_key(name: str) -> str:
    parts = name.split("_")
    return f"{parts[0]}_{parts[1]}"


def copy_split(names, split):
    (out_root / split / "img").mkdir(parents=True, exist_ok=True)
    (out_root / split / "labels").mkdir(parents=True, exist_ok=True)

    for name in names:
        shutil.copy2(img_dir / name, out_root / split / "img" / name)
        shutil.copy2(lbl_dir / name, out_root / split / "labels" / name)


def summarize_counts(names):
    return Counter(year_site_key(name) for name in names)


In [14]:
# collect files by year_site
groups = defaultdict(list)
for p in sorted(img_dir.glob("*.tif")):
    groups[year_site_key(p.name)].append(p.name)

# stratified 10% validation per year_site group
train_names = []
val_names = []

for key in sorted(groups):
    names = sorted(groups[key])
    random.shuffle(names)
    n_val = max(1, round(len(names) * VAL_FRAC))
    val_names.extend(names[:n_val])
    train_names.extend(names[n_val:])

train_names = sorted(train_names)
val_names = sorted(val_names)

train_counts = summarize_counts(train_names)
val_counts = summarize_counts(val_names)

print(f'train={len(train_names)} val={len(val_names)} total={len(train_names) + len(val_names)}')
for key in sorted(groups):
    print(key, "train=", train_counts[key], "val=", val_counts[key])


train=1895 val=212 total=2107
2018_BART train= 5 val= 1
2018_HARV train= 16 val= 2
2018_JERC train= 9 val= 1
2018_MLBS train= 45 val= 5
2018_NIWO train= 76 val= 8
2018_OSBS train= 121 val= 13
2018_SJER train= 1258 val= 140
2018_TEAK train= 291 val= 32
2019_DELA train= 14 val= 2
2019_LENO train= 18 val= 2
2019_OSBS train= 35 val= 4
2019_SJER train= 7 val= 1
2019_TOOL train= 0 val= 1


In [15]:
train_names[:10], val_names[:10]


(['2018_BART_4_322000_4882000_image_crop_patch0.tif',
  '2018_BART_4_322000_4882000_image_crop_patch1.tif',
  '2018_BART_4_322000_4882000_image_crop_patch2.tif',
  '2018_BART_4_322000_4882000_image_crop_patch4.tif',
  '2018_BART_4_322000_4882000_image_crop_patch5.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch0.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch1.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch10.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch12.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch14.tif'],
 ['2018_BART_4_322000_4882000_image_crop_patch3.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch11.tif',
  '2018_HARV_5_733000_4698000_image_crop_patch13.tif',
  '2018_JERC_4_742000_3451000_image_crop_patch7.tif',
  '2018_MLBS_3_541000_4140000_image_crop2_patch0.tif',
  '2018_MLBS_3_541000_4140000_image_crop2_patch21.tif',
  '2018_MLBS_3_541000_4140000_image_crop2_patch22.tif',
  '2018_MLBS_3_541000_4140000_image_crop_patch10.tif',
  '2018_MLBS_3_5

In [16]:
copy_split(train_names, "train")
copy_split(val_names, "val")

print(f'Wrote split to {out_root}')
print(f'train={len(train_names)} val={len(val_names)}')

Wrote split to ../data/splits/hsi_binary_10pct_balanced
train=1895 val=212


In [17]:
all_counts = Counter(year_site_key(p.name) for p in img_dir.glob("*.tif"))

print("GROUP | TOTAL | TRAIN | VAL | VAL_RATIO")
for k in sorted(all_counts):
    total = all_counts[k]
    train = train_counts[k]
    val = val_counts[k]
    ratio = val / total if total else 0
    print(f"{k:12} {total:5d} {train:5d} {val:5d} {ratio:.3f}")


GROUP | TOTAL | TRAIN | VAL | VAL_RATIO
2018_BART        6     5     1 0.167
2018_HARV       18    16     2 0.111
2018_JERC       10     9     1 0.100
2018_MLBS       50    45     5 0.100
2018_NIWO       84    76     8 0.095
2018_OSBS      134   121    13 0.097
2018_SJER     1398  1258   140 0.100
2018_TEAK      323   291    32 0.099
2019_DELA       16    14     2 0.125
2019_LENO       20    18     2 0.100
2019_OSBS       39    35     4 0.103
2019_SJER        8     7     1 0.125
2019_TOOL        1     0     1 1.000
